# Per component viz of matmus

In [1]:
import os
# Append the standard macOS TeX path to the environment
os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"

import manim

In [2]:
import mlx.core as mx
import mlx.nn as nn
from mlx.utils import tree_flatten
from mlx_lm.models.qwen3 import Model, ModelArgs

# 1. Define the 122-param Model arguments
model_args = ModelArgs(
    model_type="qwen3", hidden_size=3, num_hidden_layers=1,
    intermediate_size=3, num_attention_heads=1, rms_norm_eps=1e-6,
    vocab_size=10, tie_word_embeddings=True, num_key_value_heads=1,
    max_position_embeddings=64, rope_theta=3, head_dim=4,
)

# 2. Instantiate and load weights
model = Model(model_args)
n_params = sum(p.size for _, p in tree_flatten(model.parameters()))
checkpoint = f"checkpoint/best_{n_params}.npz"

model.load_weights(list(mx.load(checkpoint).items()))
model.eval()
mx.eval(model.parameters())
print(f"Loaded {checkpoint} ({n_params} params)\n")

# 3. Map names to instance IDs so we can track them
TARGET_MODULE_REF = {}
NAME_TO_ID = {}
for name, module in model.named_modules():
    if hasattr(module, "weight"):
        TARGET_MODULE_REF[name] = module
        NAME_TO_ID[name] = id(module)

# 4. Patch the nn.Linear class __call__ directly
SAVED_INPUTS = {}
orig_linear_call = nn.Linear.__call__

def hooked_linear_call(self, x, *args, **kwargs):
    # Save the input using the instance's unique ID
    SAVED_INPUTS[id(self)] = x
    return orig_linear_call(self, x, *args, **kwargs)

nn.Linear.__call__ = hooked_linear_call

# 5. Run a forward pass to trigger the hooks
dummy_seq = [0, 2, 1, 0, 0, 4, 3, 0] 
x = mx.array([dummy_seq], dtype=mx.int32)
_ = model(x) 

print(f"Forward pass complete!")
print("Available linear modules to visualize:")
for name, module_id in NAME_TO_ID.items():
    if module_id in SAVED_INPUTS:
        print(f" - {name}")

# --- SETUP FOR VISUALIZATION ---
TARGET_LAYER_NAME = "model.layers.0.self_attn.q_proj"
TOKEN_IDX = -1

/Users/s2011847/repos/minimal-ten-digit-addition-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded checkpoint/best_122.npz (122 params)

Forward pass complete!
Available linear modules to visualize:
 - model.layers.0.mlp.up_proj
 - model.layers.0.mlp.down_proj
 - model.layers.0.mlp.gate_proj
 - model.layers.0.self_attn.o_proj
 - model.layers.0.self_attn.v_proj
 - model.layers.0.self_attn.k_proj
 - model.layers.0.self_attn.q_proj


In [3]:
NAME_TO_ID

{'model.norm': 5045704800,
 'model.layers.0.post_attention_layernorm': 5045704720,
 'model.layers.0.input_layernorm': 5045704640,
 'model.layers.0.mlp.up_proj': 5045704560,
 'model.layers.0.mlp.down_proj': 5045704480,
 'model.layers.0.mlp.gate_proj': 5045704320,
 'model.layers.0.self_attn.k_norm': 5045703680,
 'model.layers.0.self_attn.q_norm': 5045703360,
 'model.layers.0.self_attn.o_proj': 5045703200,
 'model.layers.0.self_attn.v_proj': 5045703040,
 'model.layers.0.self_attn.k_proj': 5045701280,
 'model.layers.0.self_attn.q_proj': 5045701360,
 'model.embed_tokens': 5045701600}

# now viz!

In [4]:
from manim import *
import numpy as np

# v2

# what gemini 3.1 pro shat out

In [5]:
WAIT_CONSTANT = 4

In [6]:
# --- HELPER & BASE SCENE UPDATES ---
# (Run this cell to update the colors and the label formatting)

import numpy as np
from manim import *

def extract_attention_vectors_modified(dummy_seq=[3, 0, 4, 9]):
    NAMES = ["fluffy", "blue", "verdant", "forest"]
    x_input = mx.array([dummy_seq], dtype=mx.int32)
    _ = model(x_input)
    
    q_name = "model.layers.0.self_attn.q_proj"
    k_name = "model.layers.0.self_attn.k_proj"
    
    W_q_full = np.array(TARGET_MODULE_REF[q_name].weight)
    W_k_full = np.array(TARGET_MODULE_REF[k_name].weight)
    
    W_q = W_q_full[:3, :3].astype(float)
    W_k = W_k_full[:3, :3].astype(float)
    
    layer_inputs = np.array(SAVED_INPUTS[NAME_TO_ID[q_name]][0])
    
    vector_dict = {}
    colors = [ORANGE, PURPLE, TEAL, PINK]  
    
    for i, token_id in enumerate(dummy_seq):
        emb = layer_inputs[i, :3].astype(float)
        q_vec = W_q @ emb
        k_vec = W_k @ emb
        
        vector_dict[i] = {
            "token_id": int(token_id),
            "color": colors[i % len(colors)],
            "emb": emb,
            "q": q_vec,
            "k": k_vec,
            "name": NAMES[i]
        }
    vector_dict[3]["q"] = vector_dict[2]["k"].copy()
    vector_dict[3]["q"][2] += 1.0
    vector_dict[0]["k"] = vector_dict[0]["k"] * 2.5   # push K_fluffy out along its own ray
    vector_dict[1]["k"] = vector_dict[1]["k"] * 1.5   # push K_blue out along its own ray

        
    return vector_dict, W_q, W_k

VDATA, WQ, WK = extract_attention_vectors_modified([3, 0, 4, 9])
TARGET = 3

class Qwen3BaseScene(ThreeDScene):
    def setup_scene(self, title_text):
        self.set_camera_orientation(phi=70 * DEGREES, theta=-35 * DEGREES, zoom=1.4)
        
        box_size = 4
        self.bbox = Cube(side_length=box_size, fill_opacity=0, stroke_width=1, stroke_color=GREY_B)
        
        self.axes = ThreeDAxes(
            x_range=[-box_size/2, box_size/2, 1], y_range=[-box_size/2, box_size/2, 1], z_range=[-box_size/2, box_size/2, 1],
            x_length=box_size, y_length=box_size, z_length=box_size,
            axis_config={"color": YELLOW, "stroke_width": 3.5, "tip_width": 0.18, "tip_height": 0.22}
        )
        
        self.ghost_axes = ThreeDAxes(
            x_range=[-box_size/2, box_size/2, 1], y_range=[-box_size/2, box_size/2, 1], z_range=[-box_size/2, box_size/2, 1],
            x_length=box_size, y_length=box_size, z_length=box_size,
            axis_config={"color": YELLOW, "stroke_width": 1, "tip_width": 0.10, "tip_height": 0.12, "tick_size": 0.05}
        )

        arrow_kw = dict(thickness=0.025, height=0.22, base_radius=0.09)
        self.x_basis = Arrow3D(start=ORIGIN, end=[1, 0, 0], color=RED, **arrow_kw)
        self.y_basis = Arrow3D(start=ORIGIN, end=[0, 1, 0], color=GREEN, **arrow_kw)
        self.z_basis = Arrow3D(start=ORIGIN, end=[0, 0, 1], color=BLUE, **arrow_kw)
        
        self.label_x = Text("x", font_size=28, color=RED, weight=BOLD).move_to([1.25, 0, 0])
        self.label_y = Text("y", font_size=28, color=GREEN, weight=BOLD).move_to([0, 1.25, 0])
        self.label_z = Text("z", font_size=28, color=BLUE, weight=BOLD).move_to([0, 0, 1.25])
        
        self.add(self.bbox, self.ghost_axes, self.axes, self.x_basis, self.y_basis, self.z_basis)
        self.add_fixed_orientation_mobjects(self.label_x, self.label_y, self.label_z)

        title = Text(title_text, font_size=32, color=WHITE, weight=BOLD)
        title.to_corner(UR, buff=0.4)
        self.add_fixed_in_frame_mobjects(title)

    def create_vector(self, vec, color, dashed=False, emphasize=False):
        if dashed:
            return DashedLine(start=ORIGIN, end=vec, color=color, dash_length=0.1, stroke_width=4).add_tip(tip_width=0.15, tip_length=0.2)
        else:
            thickness = 0.04 if emphasize else 0.02
            return Arrow3D(start=ORIGIN, end=vec, color=color, thickness=thickness, height=0.18, base_radius=0.07)

    def create_label(self, vec, text, color):
        lbl = Text(f"{text}", font_size=24, color=color, weight=BOLD)
        n = np.linalg.norm(vec)
        pad = 0.3
        target_pos = vec + (pad / n) * vec if n > 1e-6 else vec
        lbl.move_to(target_pos)
        return lbl

    # --- NEW: Helper factory to keep arrows from skewing during transformations ---
    def get_dynamic_transition_helpers(self, alpha_tracker):
        def make_dynamic_arrow(start_vec, end_vec, color, emphasize=False, dashed=False):
            start_vec = np.array(start_vec, dtype=float)
            end_vec = np.array(end_vec, dtype=float)
            def get_end():
                a = alpha_tracker.get_value()
                val = (1 - a) * start_vec + a * end_vec
                if np.linalg.norm(val) < 1e-6:
                    val = np.array([1e-6, 0.0, 0.0])
                return val
            return always_redraw(lambda: self.create_vector(get_end(), color, dashed=dashed, emphasize=emphasize))

        def make_label_updater(start_vec, end_vec):
            start_vec = np.array(start_vec, dtype=float)
            end_vec = np.array(end_vec, dtype=float)
            def updater(m):
                a = alpha_tracker.get_value()
                val = (1 - a) * start_vec + a * end_vec
                n = np.linalg.norm(val)
                if n < 1e-6:
                    m.move_to(val)
                else:
                    m.move_to(val + (0.3 / n) * val)
            return updater
            
        return make_dynamic_arrow, make_label_updater

## 0. 1 token scene

In [ ]:
%%manim -v WARNING -qh A_ShowOneEmbedding

class A_ShowOneEmbedding(Qwen3BaseScene):
    def construct(self):
        self.setup_scene("1. Tokens")
        
        arrows, labels = [], []
        for i, data in VDATA.items():
            arr = self.create_vector(data["emb"], data["color"])
            
            label_text = data["name"]
                
            lbl = self.create_label(data["emb"], label_text, data["color"])
            arrows.append(arr)
            labels.append(lbl)
            break
            
        self.begin_ambient_camera_rotation(rate=0.3)
        self.wait(2)
            
        self.play(*[GrowFromPoint(arr, ORIGIN) for arr in arrows], run_time=1.5)
        self.add_fixed_orientation_mobjects(*labels)
        
        self.wait(WAIT_CONSTANT)
        self.stop_ambient_camera_rotation()

Manim Community v0.20.1

## 1. All Tokens scene

In [8]:
%%manim -v WARNING -ql A_ShowEmbeddings

class A_ShowEmbeddings(Qwen3BaseScene):
    def construct(self):
        self.setup_scene("1. Tokens")
        
        arrows, labels = [], []
        for i, data in VDATA.items():
            arr = self.create_vector(data["emb"], data["color"])
            
            label_text = data["name"]
                
            lbl = self.create_label(data["emb"], label_text, data["color"])
            arrows.append(arr)
            labels.append(lbl)
            
        self.begin_ambient_camera_rotation(rate=0.3)
        self.wait(2)
            
        self.play(*[GrowFromPoint(arr, ORIGIN) for arr in arrows], run_time=1.5)
        self.add_fixed_orientation_mobjects(*labels)
        
        self.wait(WAIT_CONSTANT)
        self.stop_ambient_camera_rotation()

Manim Community v0.20.1

## 2. Queries scene

In [9]:
%%manim -v WARNING -ql B_ShowQueryProj

class B_ShowQueryProj(Qwen3BaseScene):
    def construct(self):
        self.setup_scene("2. Queries")
        
        arrows, labels = [], []
        for i, data in VDATA.items():
            arr = self.create_vector(data["emb"], data["color"], emphasize=(i == TARGET))
            
            label_text = data["name"]
                
            lbl = self.create_label(data["emb"], label_text, data["color"])
            arrows.append(arr)
            labels.append(lbl)
            
        self.add(*arrows)
        self.add_fixed_orientation_mobjects(*labels)
        self.wait(1)
        
        # Fade non-targets
        non_target_anims = []
        for i, arr in enumerate(arrows):
            if i != TARGET:
                non_target_anims.append(arr.animate.set_opacity(0.01))
                non_target_anims.append(labels[i].animate.set_opacity(0.01))
        
        self.play(*non_target_anims)
        self.wait(1)
        
        # --- DYNAMIC MATRIX TRANSFORMATION ---
        alpha = ValueTracker(0.0)
        make_arrow, make_updater = self.get_dynamic_transition_helpers(alpha)
        
        # 1. Transform RGB Basis Vectors
        end_x = WQ @ np.array([1.0, 0.0, 0.0])
        end_y = WQ @ np.array([0.0, 1.0, 0.0])
        end_z = WQ @ np.array([0.0, 0.0, 1.0])
        
        dyn_x = make_arrow([1, 0, 0], end_x, RED)
        dyn_y = make_arrow([0, 1, 0], end_y, GREEN)
        dyn_z = make_arrow([0, 0, 1], end_z, BLUE)
        
        self.remove(self.x_basis, self.y_basis, self.z_basis)
        self.add(dyn_x, dyn_y, dyn_z)
        
        self.label_x.add_updater(make_updater([1, 0, 0], end_x))
        self.label_y.add_updater(make_updater([0, 1, 0], end_y))
        self.label_z.add_updater(make_updater([0, 0, 1], end_z))

        # 2. Transform the Target Token Vector
        target_emb = VDATA[TARGET]["emb"]
        target_q = VDATA[TARGET]["q"]
        target_color = VDATA[TARGET]["color"]
        
        dyn_target = make_arrow(target_emb, target_q, target_color, emphasize=True)
        self.remove(arrows[TARGET])
        self.add(dyn_target)
        
        labels[TARGET].add_updater(make_updater(target_emb, target_q))
        
        # Run Animation
        self.play(
            ApplyMatrix(WQ, self.axes),
            alpha.animate.set_value(1.0),
            run_time=3
        )
        
        # Cleanup updaters
        self.label_x.clear_updaters()
        self.label_y.clear_updaters()
        self.label_z.clear_updaters()
        labels[TARGET].clear_updaters()
        
        new_q_lbl = self.create_label(VDATA[TARGET]["q"], "Q (forest)", VDATA[TARGET]["color"])
        self.add_fixed_orientation_mobjects(new_q_lbl)
        self.play(FadeOut(labels[TARGET]), FadeIn(new_q_lbl))
        labels[TARGET] = new_q_lbl
        
        self.begin_ambient_camera_rotation(rate=0.3)
        self.wait(WAIT_CONSTANT)
        self.stop_ambient_camera_rotation()

Manim Community v0.20.1

## 3. Keys scene

In [10]:
%%manim -v WARNING -ql C_ShowKeyProj

class C_ShowKeyProj(Qwen3BaseScene):
    def construct(self):
        self.setup_scene("3. Keys")
        
        arrows, labels = [], []
        for i, data in VDATA.items():
            if data["name"] == "forest":
                continue
            arr = self.create_vector(data["emb"], data["color"])
            
            label_text = data["name"]
            
            lbl = self.create_label(data["emb"], label_text, data["color"])
            arrows.append(arr)
            labels.append(lbl)
            
        self.add(*arrows)
        self.add_fixed_orientation_mobjects(*labels)
        self.wait(1)
        
        # --- DYNAMIC MATRIX TRANSFORMATION ---
        alpha = ValueTracker(0.0)
        make_arrow, make_updater = self.get_dynamic_transition_helpers(alpha)
        
        # 1. Transform RGB Basis Vectors
        end_x = WK @ np.array([1.0, 0.0, 0.0])
        end_y = WK @ np.array([0.0, 1.0, 0.0])
        end_z = WK @ np.array([0.0, 0.0, 1.0])
        
        dyn_x = make_arrow([1, 0, 0], end_x, RED)
        dyn_y = make_arrow([0, 1, 0], end_y, GREEN)
        dyn_z = make_arrow([0, 0, 1], end_z, BLUE)
        
        self.remove(self.x_basis, self.y_basis, self.z_basis)
        self.add(dyn_x, dyn_y, dyn_z)
        
        self.label_x.add_updater(make_updater([1, 0, 0], end_x))
        self.label_y.add_updater(make_updater([0, 1, 0], end_y))
        self.label_z.add_updater(make_updater([0, 0, 1], end_z))

        # 2. Transform ALL Token Vectors
        dyn_arrows = []
        for i, arr in enumerate(arrows):
            emb = VDATA[i]["emb"]
            k_vec = VDATA[i]["k"]
            color = VDATA[i]["color"]
            
            dyn_arr = make_arrow(emb, k_vec, color)
            dyn_arrows.append(dyn_arr)
            
            self.remove(arr)
            labels[i].add_updater(make_updater(emb, k_vec))
            
        self.add(*dyn_arrows)
            
        # Run Animation
        self.play(
            ApplyMatrix(WK, self.axes),
            alpha.animate.set_value(1.0),
            run_time=3
        )
        
        # Cleanup updaters
        self.label_x.clear_updaters()
        self.label_y.clear_updaters()
        self.label_z.clear_updaters()
        for lbl in labels:
            lbl.clear_updaters()
            
        kept_keys = [i for i, data in VDATA.items() if data["name"] != "forest"]
        new_labels = []
        anims = []
        for arr_idx, vkey in enumerate(kept_keys):
            name = VDATA[vkey]["name"]
            new_lbl = self.create_label(VDATA[vkey]["k"], f"K ({name})", VDATA[vkey]["color"])
            self.add_fixed_orientation_mobjects(new_lbl)
            new_labels.append(new_lbl)
            anims.append(FadeOut(labels[arr_idx]))
            anims.append(FadeIn(new_lbl))
        self.play(*anims)
        for arr_idx, new_lbl in enumerate(new_labels):
            labels[arr_idx] = new_lbl
            
        self.begin_ambient_camera_rotation(rate=0.3)
        self.wait(WAIT_CONSTANT)
        self.stop_ambient_camera_rotation()

Manim Community v0.20.1

## 4. Attention Scene

In [11]:
%%manim -v WARNING -ql D_ShowAttention

class D_ShowAttention(Qwen3BaseScene):
    def construct(self):
        self.setup_scene("4. Attention")
        
        self.begin_ambient_camera_rotation(rate=0.4)
        
        q_data = VDATA[TARGET]
        q_arr = self.create_vector(q_data["q"], q_data["color"], emphasize=True)
        q_lbl = self.create_label(q_data["q"], f"Q (forest)", q_data["color"])
        
        self.add(q_arr)
        self.add_fixed_orientation_mobjects(q_lbl)
        
        k_arrows, k_labels = [], []
        kept_keys = []
        for i, data in VDATA.items():
            if data["name"] == "forest":
                continue
            k_arr = self.create_vector(data["k"], data["color"], dashed=True)
            k_lbl = self.create_label(data["k"], f"K ({data['name']})", data["color"])
            k_arrows.append(k_arr)
            k_labels.append(k_lbl)
            kept_keys.append(i)
            
        self.play(*[Create(k) for k in k_arrows], run_time=1.5)
        self.add_fixed_orientation_mobjects(*k_labels)
        self.wait(1)
        
        dist_lines = []
        shortest_dist = float('inf')
        best_match_idx = -1
        
        for arr_idx, vkey in enumerate(kept_keys):
            q_vec = q_data["q"]
            k_vec = VDATA[vkey]["k"]
            
            # Euclidean distance (proxy for attention score in this viz)
            dist = np.linalg.norm(q_vec - k_vec)
            
            if dist < shortest_dist:
                shortest_dist = dist
                best_match_idx = arr_idx
                
            line = DashedLine(q_vec, k_vec, color=WHITE, stroke_width=2, dash_length=0.05).set_opacity(0.6)
            dist_lines.append(line)
            
        self.play(*[Create(line) for line in dist_lines])
        self.wait(1)
        
        # Highlight the shortest distance
        self.play(
            dist_lines[best_match_idx].animate.set_color(RED).set_stroke(width=16).set_opacity(1.0),
            k_arrows[best_match_idx].animate.set_color(RED)
        )
        
        self.wait(WAIT_CONSTANT)
        self.stop_ambient_camera_rotation()

Manim Community v0.20.1